In [1]:
from oo_cqed_rhf import CQEDRHFCalculator
import numpy as np
import psi4
import sys
from typing import TextIO  # ← add this import
psi4.core.be_quiet()

In [2]:
def bfgs_update(xk, gk, xkp1, gkp1, Hk):
    """
    Performs a BFGS update of the Hessian approximation.  Needs two solution vectors (xk and xk+1)
    and their corresponding gradients (gk and gk+1) and the current Hessian approximation (Hk).

    Arguments
    ---------
    xk : numpy matrix representation of a row vector (1x2 matrix)
       the previous solution vector

    gk : numpy matrix representation of a row vector (1x2 matrix)
       the gradient at the previous solution vector

    xkp1 : numpy matrix representation of a row vector (1x2 matrix)
       the next solution vector

    gkp1 : numpy matrix representation of a row vector (1x2 matrix)
       the gradient at the next solution vector

    Hk : numpy matrix representatio of the Hessian (2x2 matrix)
       the current Hessian approximation


    Returns
    --------
    Hkp1 : numpy matrix representatio of the Hessian (2x2 matrix)
       the updated Hessian approximation
    """

    # compute yk
    yk = np.matrix(gkp1 - gk).T

    # compute sk
    sk = np.matrix(xkp1 - xk).T

    # compute alpha denominator for Hessian update
    a_d = yk.T @ sk

    # treat a_d as a scalar
    alpha = 1 / a_d[0,0]


    # compute beta denominator for Hessian update
    b_d = sk.T @ Hk @ sk

    # treat b_d as a scalr
    beta = - 1 / b_d[0,0]


    # Check if the dot product of alpha are beta are close to zero - these will make the Hessian update large
    if np.isclose(alpha, 0):
      print("Error: The dot product of y and s is close to zero. BFGS update is not performed.")
      return Hk

    if np.isclose(beta, 0):
      print("Error: The dot product of s and Hk and sk is close to zero. BFGS update is not performed.")
      return Hk

    # first part of update
    B1 = alpha * yk @ yk.T

    # second part of update
    B2 = beta * Hk @ sk @ sk.T @ Hk.T

    # update to Hessian
    Hkp1 = Hk + B1 + B2

    return Hkp1

def geom_bohr_to_angstrom_string(geom_bohr: np.ndarray,
                                 symbols: list[str]) -> str:
    """
    Convert an (N,3) array in Bohr to a psi4 geometry input string in Angstrom,
    appending the fixed Psi4 directives at the end.

    Returns a string like:
        O   x   y   z
        H   x   y   z
        -1 1
        no_reorient
        nocom
        symmetry c1
    """
    # Conversion factor: 1 bohr = 0.529177210903 Å
    bohr2ang = 0.529177210903

    # Convert coordinates
    geom_ang = geom_bohr * bohr2ang

    # Format atomic lines
    lines = []
    for sym, (x, y, z) in zip(symbols, geom_ang):
        lines.append(f"{sym:2s}  {x:14.12f}  {y:14.12f}  {z:14.12f}")

    # Append fixed Psi4 directives
    suffix = """
0 1
no_reorient
nocom
symmetry c1
""".strip()

    # Combine and return full input block
    return "\n".join(lines + [suffix])


def optimize_geometry(calc,
                      x0_bohr: np.ndarray,      # shape (N_atoms,3)
                      symbols: list[str],
                      bfgs_update,
                      tol: float = 1e-6,
                      max_iter: int = 50):
    """
    BFGS‐style geometry optimizer using your calc.calc_force_and_energy().
    Returns the converged x (Bohr coords), E, gradient, and Hessian.
    """
    n_atoms = x0_bohr.shape[0]
    # total degrees of freedom
    ndim = 3 * n_atoms

    # flatten the initial geometry
    xk = x0_bohr.reshape(ndim)

    # initial Hessian guess (identity)
    Hk = np.eye(ndim)

    # initial energy & gradient
    geom_block = geom_bohr_to_angstrom_string(xk.reshape(n_atoms,3), symbols)
    Ek, gk_full, g = calc.calc_force_and_energy(
        geom_block, use_psi4_scf_grad=False
    )
    # flatten gradient
    gk = gk_full.reshape(ndim)

    converged = False
    for iteration in range(1, max_iter+1):
        # 1) Compute step pk = - Hk^{-1} gk
        pk = -np.linalg.solve(Hk, gk)

        # 2) Update geometry
        xkp1 = xk + pk

        # 3) Evaluate energy & gradient at new x
        geom_block = geom_bohr_to_angstrom_string(
            xkp1.reshape(n_atoms,3), symbols
        )
        Ekp1, gkp1_full, g = calc.calc_force_and_energy(
            geom_block, use_psi4_scf_grad=False
        )
        gkp1 = gkp1_full.reshape(ndim)

        # 4) Check convergence
        norm_g = np.linalg.norm(gkp1)
        print(f"Iter {iteration:2d}: E = {Ekp1:.8f} Eh   ‖g‖ = {norm_g:.2e}")
        if norm_g < tol:
            converged = True
            break

        # 5) BFGS update of H
        #    we assume bfgs_update accepts flattened vectors
        Hkp1 = bfgs_update(xk, gk, xkp1, gkp1, Hk)

        # 6) Shift these for next iter
        xk, gk, Hk, Ek = xkp1, gkp1, Hkp1, Ekp1

    # end for

    # Final report
    if converged:
        print("\nConverged!")
    else:
        print("\nWARNING: max_iter reached without convergence.")

    final_geom = geom_bohr_to_angstrom_string(xk.reshape(n_atoms,3), symbols)
    print(f"Final geometry (Bohr):\n{xk.reshape(n_atoms,3)}")
    print(f"Final energy: {Ekp1:.8f} Eh")
    print(f"Final gradient:\n{gkp1_full}")
    print("Final Geometry String:\n")
    print(final_geom)
    return xk.reshape(n_atoms,3), Ekp1, gkp1_full, Hk

import sys
import numpy as np

# conversion factor
BOHR_TO_ANGSTROM = 0.52917721092

def velocity_verlet_step(x_bohr: np.ndarray,
                         v_bohr: np.ndarray,
                         grad: np.ndarray,
                         masses: np.ndarray,
                         dt: float,
                         calc,
                         symbols: list[str],
                         frame: int,
                         use_psi4_scf_grad: bool = False,
                         out: TextIO = sys.stdout):
    """
    Perform one Velocity-Verlet step *and* print an XYZ frame for VMD.

    Args:
        x_bohr   : (N,3) positions in Bohr
        v_bohr   : (N,3) velocities in Bohr/atomic-time
        grad     : (N,3) gradient in Hartree / Bohr
        masses   : (N,) masses in atomic electron-mass units
        dt       : time step (atomic units of time)
        calc     : object with calc_force_and_energy() → (E, grad)
        symbols  : list of atomic symbols
        frame    : integer frame index (for the comment line)
        use_psi4_scf_grad: whether to use psi4's SCF gradient
        out      : file-like to write the XYZ data (defaults to stdout)

    Returns:
        x_new    : (N,3) updated positions (Bohr)
        v_new    : (N,3) updated velocities
        E_new    : energy at new positions
        grad_new : gradient at new positions
        g_new    : effective coupling strength at updated position
    """
    n_atoms = x_bohr.shape[0]


    F_old = -grad # gradient passed to function
    a_old = F_old / masses[:, None]


        
    # --- propagate positions ---
    x_new = x_bohr + v_bohr * dt + 0.5 * a_old * dt**2



    # --- new forces & accelerations ---
    geom_block_new = geom_bohr_to_angstrom_string(x_new, symbols)
    E_new, grad_new, g_new = calc.calc_force_and_energy(
        geom_block_new, use_psi4_scf_grad=use_psi4_scf_grad
    )
    F_new = -grad_new

    a_new = F_new / masses[:, None]



    # --- propagate velocities ---
    v_new = v_bohr + 0.5 * (a_old + a_new) * dt

        
    # --- print XYZ frame to `out` ---
    # 1) number of atoms
    print(n_atoms, file=out)
    # 2) comment line: frame index, energy in Hartree
    print(f"Frame {frame}   E = {E_new:.8f} Ha", file=out)
    # 3) atom lines, converting Bohr→Å
    coords_ang = x_new * BOHR_TO_ANGSTROM
    for sym, (x, y, z) in zip(symbols, coords_ang):
        print(f"{sym:<2} {x: .6f} {y: .6f} {z: .6f}", file=out)

    return x_new, v_new, E_new, grad_new, g_new



In [3]:

# Global Constants (Atomic Units conversion)
fs_timeau = 41.34137314
amu2au = 1822.8884850


# lambda vector along z
lambda_vector = np.array([0, 0.05, 0.05])


# psi4 options
psi4_options = {
    "basis": "6-31G",
    "save_jk": True,
    "scf_type": "pk",
    "e_convergence": 1e-12,
    "d_convergence": 1e-12,
}


psi4.set_options(psi4_options)

## forward displaced geometry string
mol_string = """
0  1
Li    0.000000000000    0.000000000000   0
H    0.000000000000    0.000000000000    1.5
no_reorient
nocom
symmetry c1
"""

mol = psi4.geometry(mol_string)
e = psi4.energy("scf")
print(f"RHF energy is {e:6.12f}")
natoms = mol.natom()
atom_mass = np.asarray([mol.mass(atom) for atom in range(natoms)])*amu2au

atom_mass = np.array([12789.3918753,  1837.17993072])
print(atom_mass)

print(mol.geometry().to_array())
x0_bohr = mol.geometry().to_array()

# get atomic symbols from geometry
symbols = [mol.symbol(i) for i in range(mol.natom())]  # ["O","H"]


print(x0_bohr)



RHF energy is -7.976859130046
[12789.3918753   1837.17993072]
[[0.         0.         0.        ]
 [0.         0.         2.83458919]]
[[0.         0.         0.        ]
 [0.         0.         2.83458919]]


In [4]:
# we will pass our desired geometry string when we want to compute the gradient
calc = CQEDRHFCalculator(lambda_vector, mol_string, psi4_options)

# calculate the CQED-RHF energy and gradient at h2o_string_b, use our routines for all terms
qed_rhf_energy, qed_rhf_grad, qed_rhf_g = calc.calc_force_and_energy(mol_string, use_psi4_scf_grad=False)
print(F"QED-RHF Energy initial is {qed_rhf_energy:6.12f}")
print(F"QED-RHF Grad Initial is \n {qed_rhf_grad}")

Not Using Density Fitting!
QED-RHF Energy initial is -7.968007513105
QED-RHF Grad Initial is 
 [[ 1.01444990e-17  4.84013354e-04  2.00289880e-02]
 [-1.01444990e-17 -4.84013354e-04 -2.00289880e-02]]


In [5]:
dt = 15
x_curr = np.copy(x0_bohr)
v_curr = np.array([
    [ 0.0,   4e-4/12789.3918753,  0.000],
    [0.0,  -4e-4/1837.15264615,   0.00]
])

print(x0_bohr)
print(qed_rhf_grad)
print(v_curr)

#v_curr = np.zeros_like(x_curr)


n_steps = 150
sE_list = []
sg_list = []
with open("lih_traj_ruby_conditions_dt_15_lz_0.05_no_df.xyz", "w") as traj:
    x = x_curr          # initial positions in Bohr
    v = v_curr          # initial velocities
    grad = qed_rhf_grad # initial gradient
    for i in range(n_steps):
        x, v, E, grad, g_new = velocity_verlet_step(
            x, v, grad, atom_mass, dt, calc, symbols,
            frame=i,                      # frame counter
            use_psi4_scf_grad=False,
            out=traj                      # write into traj.xyz
        )
        sE_list.append(E)
        sg_list.append(g_new)
#Once you’ve generated traj.xyz, you can load it directly into VMD (
#File → New Molecule → select traj.xyz, set file type to “XYZ” and click “Load” ) and then hit Play to see your dynamics.




[[0.         0.         0.        ]
 [0.         0.         2.83458919]]
[[ 1.01444990e-17  4.84013354e-04  2.00289880e-02]
 [-1.01444990e-17 -4.84013354e-04 -2.00289880e-02]]
[[ 0.00000000e+00  3.12759202e-08  0.00000000e+00]
 [ 0.00000000e+00 -2.17728233e-07  0.00000000e+00]]
x(98)

[[-1.82278004e-18 -2.85819935e-02 -6.22511498e-02]
 [ 1.26891481e-17  1.98971424e-01  3.26794595e+00]]
v(98)

[[ 8.29841364e-20 -3.03774449e-05  3.14043170e-05]
 [-5.77687913e-19  2.11470327e-04 -2.18618824e-04]]
a(98)

[[-1.49660230e-21  6.82179408e-08  1.17350259e-06]
 [ 1.04184860e-20 -4.74894137e-07 -8.16925127e-06]]
x(99)

[[-7.46385754e-19 -2.90299807e-02 -6.16480660e-02]
 [ 5.19590909e-18  2.02090053e-01  3.26374763e+00]]
v(99)

[[ 7.79560916e-20 -2.93534104e-05  4.88900460e-05]
 [-5.42685552e-19  2.04341587e-04 -3.40344430e-04]]
a(99)

[[ 8.26196325e-22  6.83199954e-08  1.15792794e-06]
 [-5.75150446e-21 -4.75604583e-07 -8.06082952e-06]]
x(100)

[[ 5.15902707e-19 -2.94625958e-02 -6.07844484e-02]
 [

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# … assume t_list, E_list, and E_rhf are already defined …

t_list = np.linspace(0, dt * 999, 1000)

# Make a larger figure
fig, ax = plt.subplots(figsize=(10, 6))

# Plot the coupled energy
ax.plot(
    t_list, 
    E_list, 
    linewidth=2, 
    label='E$_{QEDRHF}$'
)

# Optionally, show the reference E_rhf as a horizontal line
ax.axhline(
    E_rhf, 
    color='red', 
    linestyle='--', 
    linewidth=1.5, 
    label='E$_{RHF}$'
)

# Labels and title
ax.set_xlabel('Time (a.u.)', fontsize=14)
ax.set_ylabel('Energy (Hartree)', fontsize=14)
ax.set_title('Coupled Energy vs Time', fontsize=16, pad=15)

# Legend
ax.legend(frameon=False, fontsize=12)

# Tweak layout so things fit nicely
plt.tight_layout()

plt.show()
# E_rhf in Hartrees
E_rhf = -75.383352863532

# E list in Hartrees
E_array = np.array(E_list)
max_coupled_E = np.max(E_array)
min_coupled_E = np.min(E_array)
barrier_height = max_coupled_E - min_coupled_E

max_cavity_effect = max_coupled_E - E_rhf
min_cavity_effect = min_coupled_E - E_rhf

# Conversion factor
hartree_to_ev = 27.211386 

# Convert to eV
barrier_height_ev    = barrier_height    * hartree_to_ev
max_cavity_effect_ev = max_cavity_effect * hartree_to_ev
min_cavity_effect_ev = min_cavity_effect * hartree_to_ev

# Print nicely
print(f"Initial RHF Energy    {E_rhf:12.6f} Ha")
print(f"Initial QED-HF Energy {qed_rhf_energy:12.6f} Ha")
print(f"Barrier height:       {barrier_height_ev:12.6f} eV")
print(f"Max cavity effect:    {max_cavity_effect_ev:12.6f} eV")
print(f"Min cavity effect:    {min_cavity_effect_ev:12.6f} eV")

In [ ]:
plt.plot(t_list, np.abs(g_list)+np.mean(np.abs(E_list))/1.008)
plt.plot(t_list, np.abs(E_list))
plt.plot(t_list, np.abs(sg_list)+np.mean(np.abs(sE_list))/1.008)
plt.plot(t_list, np.abs(sE_list))
plt.show()

In [ ]:
optimize_geometry(calc, x0_bohr, symbols, bfgs_update, tol=1e-7, max_iter=50)



In [ ]:
# initial step for BFGS update

# initialize Hessian
n_atoms = 2

Hi = np.eye(3 * n_atoms)

# initialize gradient in atomic units
gi = qed_rhf_grad

# initialize energy
Ei = qed_rhf_energy

# initial update in atomic units
pi = -np.linalg.inv(Hi) @ gi

# initial geometry in atomic units
xi = mol.geometry().to_array()

# update geometry
x = xi + pi

# update mol_string and get new gradient
mol_string = geom_bohr_to_angstrom_string(x, symbols)

# calculate the CQED-RHF energy and gradient at h2o_string_b, use our routines for all terms
qed_rhf_energy, qed_rhf_grad, qed_rhf_g = calc.calc_force_and_energy(mol_string, use_psi4_scf_grad=False)

# get new gradient and energy
g = qed_rhf_grad
E = qed_rhf_energy


# prepare for BFGS updates
xk = np.copy(xi)
xkp1 = np.copy(x)
gk = np.copy(gi)
gkp1 = np.copy(g)
Hk = np.copy(Hi)


converged = False
for i in range(10):
    Hkp1 = bfgs_update(xk, gk, xkp1, gkp1, Hk)
    
    #print(F"Updated Hessian is")
    #print(Hkp1)
    
    # update solutions xk and xk+1
    xk = np.copy(xkp1)
    
    #print(F"solution is {xk}")
    
    pk = -np.linalg.inv(Hkp1) @ gkp1
    
    #print(F"Update is {pk}")
    
    xkp1 = xk + pk
    print(F"Updated solution is {xkp1[0,:]}")
    
    # update gradients gk and gk+1
    gk = np.copy(gkp1)
    # update mol_string and get new gradient
    mol_string = geom_bohr_to_angstrom_string(xkp1, symbols)
    
    # calculate the CQED-RHF energy and gradient at h2o_string_b, use our routines for all terms
    qed_rhf_energy, qed_rhf_grad, qed_rhf_g = calc.calc_force_and_energy(mol_string, use_psi4_scf_grad=False)
    
    # get new gradient and energy
    gkp1 = qed_rhf_grad
    Ekp1 = qed_rhf_energy

    
    # compute norm of gradient
    norm_grad = np.linalg.norm(gkp1)
    print(F"Norm of gradient is {norm_grad}")
    if norm_grad < 1e-6:
        converged = True
        break
    else:
        Hk = np.copy(Hkp1)
    
    if converged:
        print("\n\n\nConverged!!!!")
        print(F"Final solution is {xkp1[0,:]}")
        print(F"Final gradient is {gkp1}")
        print(F"Final Hessian is")
        print(Hk)
    else:
        print("\n\n\nDid not converge")
        print(F"Final solution is {xkp1[0,:]}")
        print(F"Final gradient is {gkp1}")
        print(F"Final Hessian is")
        print(Hk)
